# OpenAQ Data Loader for ML

Single unified method to load, prepare, and split PM2.5 data for machine learning projects.

## Setup

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Tuple, Dict, Optional

np.random.seed(42)
pd.set_option('display.max_columns', None)

print("✓ Libraries imported")

✓ Libraries imported


## Unified Data Loader

In [23]:
class OpenAQMLDataLoader:
    """Single unified loader for PM2.5 data preparation."""
    
    def __init__(self, csv_file: str = 'openaq_ground_truth.csv'):
        """Initialize loader with CSV file path."""
        self.csv_file = csv_file
        self.df = None
        self.X_train = None
        self.y_train = None
        self.X_test = None
        self.y_test = None
        self.feature_names = None
        self.train_df = None
        self.test_df = None
    
    def load_and_prepare(self, test_size: float = 0.2) -> Dict:
        """Load data, clean, engineer features, and create train/test split.
        
        Args:
            test_size: Fraction of data for testing (default: 0.2)
            
        Returns:
            Dictionary with X_train, y_train, X_test, y_test, feature_names
        """
        # Step 1: Load
        self._load()
        print(f"✓ Loaded {len(self.df):,} measurements")
        
        # Step 2: Clean
        self._clean()
        print(f"✓ Cleaned data: {len(self.df):,} valid measurements")
        
        # Step 3: Engineer features
        self._engineer_features()
        print(f"✓ Features engineered: {len(self.df.columns)} columns")
        
        # Step 4: Split
        self._train_test_split(test_size)
        print(f"✓ Train/test split: {len(self.train_df):,} train, {len(self.test_df):,} test")
        
        # Step 5: Create feature matrices
        self._create_feature_matrices()
        print(f"✓ Feature matrices created: {self.X_train.shape}")
        
        return self.get_data()
    
    def _load(self):
        """Load CSV file."""
        if not Path(self.csv_file).exists():
            raise FileNotFoundError(
                f"{self.csv_file} not found.\n"
                f"1. Create data with OpenAQ API:\n"
                f"   from openaq import OpenAQ\n"
                f"   api = OpenAQ()\n"
                f"   df = api.measurements(city='Beijing', parameter='pm25', df=True)\n"
                f"   df.to_csv('{self.csv_file}', index=False)\n"
                f"2. Or download from explore.openaq.org"
            )
        
        self.df = pd.read_csv(self.csv_file)
    
    def _clean(self):
        """Clean and validate data."""
        # Remove duplicates
        self.df = self.df.drop_duplicates()
        
        # Ensure value is numeric
        if 'value' in self.df.columns:
            self.df['value'] = pd.to_numeric(self.df['value'], errors='coerce')
        
        # Remove null values in target
        self.df = self.df.dropna(subset=['value'])
        
        # Convert date to datetime
        if 'date' in self.df.columns:
            self.df['date'] = pd.to_datetime(self.df['date'], errors='coerce')
    
    def _engineer_features(self):
        """Create ML features."""
        # Temporal features
        if 'date' in self.df.columns:
            self.df['year'] = self.df['date'].dt.year
            self.df['month'] = self.df['date'].dt.month
            self.df['day'] = self.df['date'].dt.day
            self.df['dayofweek'] = self.df['date'].dt.dayofweek
            self.df['quarter'] = self.df['date'].dt.quarter
            self.df['is_weekend'] = self.df['dayofweek'].isin([5, 6]).astype(int)
            
            # Season: 0=Winter, 1=Spring, 2=Summer, 3=Fall
            seasons = [0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 0]
            self.df['season'] = self.df['month'].map(lambda x: seasons[x-1])
        
        # Region one-hot encoding
        if 'region' in self.df.columns:
            region_dummies = pd.get_dummies(self.df['region'], prefix='region')
            self.df = pd.concat([self.df, region_dummies], axis=1)
        
        # Target transformations
        self.df['value_log'] = np.log1p(self.df['value'])
        
        # AQI category
        def categorize(v):
            if v <= 12: return 0
            elif v <= 35.4: return 1
            elif v <= 55.4: return 2
            elif v <= 150.4: return 3
            elif v <= 250.4: return 4
            else: return 5
        self.df['aqi_category'] = self.df['value'].apply(categorize)
    
    def _train_test_split(self, test_size: float):
        """Create train/test split."""
        mask = np.random.rand(len(self.df)) < (1 - test_size)
        self.train_df = self.df[mask].reset_index(drop=True)
        self.test_df = self.df[~mask].reset_index(drop=True)
    
    def _create_feature_matrices(self):
        """Create numpy arrays for ML models."""
        exclude = {
            'value', 'aqi_category', 'value_log',
            'date', 'location', 'city', 'country', 'region',
            'unit', 'parameter', 'coordinates', 'attribution'
        }
        
        self.feature_names = [
            col for col in self.train_df.columns
            if col not in exclude and self.train_df[col].dtype in ['float64', 'int64']
        ]
        
        self.X_train = self.train_df[self.feature_names].values
        self.y_train = self.train_df['value'].values
        self.X_test = self.test_df[self.feature_names].values
        self.y_test = self.test_df['value'].values
    
    def get_data(self) -> Dict:
        """Return all prepared data."""
        return {
            'X_train': self.X_train,
            'y_train': self.y_train,
            'X_test': self.X_test,
            'y_test': self.y_test,
            'feature_names': self.feature_names,
            'train_df': self.train_df,
            'test_df': self.test_df
        }
    
    def get_summary(self) -> str:
        """Get data summary."""
        summary = f"""
        Dataset Summary:
        ===============
        Training samples: {len(self.train_df):,}
        Test samples: {len(self.test_df):,}
        Features: {len(self.feature_names)}
        
        PM2.5 Stats:
          Mean: {self.y_train.mean():.2f} µg/m³
          Median: {np.median(self.y_train):.2f} µg/m³
          Std: {self.y_train.std():.2f} µg/m³
          Min: {self.y_train.min():.2f} µg/m³
          Max: {self.y_train.max():.2f} µg/m³
        """
        return summary


print("✓ OpenAQMLDataLoader class defined")

✓ OpenAQMLDataLoader class defined


## Load Data - Single Call

In [24]:
# Define path to data folder
data_path = Path('../data/openaq_ground_truth.csv')

# Create sample data if file doesn't exist
if not data_path.exists():
	# Generate 10,000 PM2.5 measurements from Beijing across 2 years
	n_samples = 10000
	sample_data = {
		'date': pd.date_range('2022-01-01', periods=n_samples, freq='6H'),
		'value': np.abs(np.random.normal(loc=65, scale=35, size=n_samples)),  # PM2.5 values with realistic distribution
		'region': np.random.choice(['Chaoyang', 'Haidian', 'Fengtai', 'Shunyi', 'Daxing'], n_samples),
		'location': 'Beijing CBD',
		'city': 'Beijing',
		'country': 'China',
		'unit': 'µg/m³',
		'parameter': 'pm25'
	}
	df_sample = pd.DataFrame(sample_data)
	
	# Create data directory if it doesn't exist
	data_path.parent.mkdir(parents=True, exist_ok=True)
	
	# Save to CSV
	df_sample.to_csv(data_path, index=False)
	print(f"✓ Sample data created: {data_path}")
	print(f"  - {n_samples:,} PM2.5 measurements from Beijing")
	print(f"  - Date range: {df_sample['date'].min()} to {df_sample['date'].max()}")
	print(f"  - PM2.5 range: {df_sample['value'].min():.1f} - {df_sample['value'].max():.1f} µg/m³")
else:
	csv_df = pd.read_csv(data_path)
	print(f"✓ Data file found: {data_path}")
	print(f"  - {len(csv_df):,} measurements loaded")
	print(f"  - Date range: {csv_df['date'].min()} to {csv_df['date'].max()}")

# Initialize and load
loader = OpenAQMLDataLoader(csv_file=str(data_path))

# All-in-one: Load, clean, engineer features, split, and prepare for ML
data = loader.load_and_prepare(test_size=0.2)

# Extract from returned dictionary
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']
feature_names = data['feature_names']
train_df = data['train_df']
test_df = data['test_df']

print(loader.get_summary())

✓ Data file found: ..\data\openaq_ground_truth.csv
  - 500 measurements loaded
  - Date range: 2023-01-01 00:00:00 to 2023-01-21 19:00:00
✓ Loaded 500 measurements
✓ Cleaned data: 500 valid measurements
✓ Features engineered: 21 columns
✓ Train/test split: 394 train, 106 test
✓ Feature matrices created: (394, 2)

        Dataset Summary:
        Training samples: 394
        Test samples: 106
        Features: 2

        PM2.5 Stats:
          Mean: 29.25 µg/m³
          Median: 25.17 µg/m³
          Std: 19.92 µg/m³
          Min: 1.98 µg/m³
          Max: 101.95 µg/m³
        
